# Nikash — Kaggle run (pilot and full run)

Runs on a free Kaggle GPU. Settings: **Accelerator = GPU T4 x2** (one model copy per GPU; a single T4 also works, at half the speed), **Internet = on**. Set `MODE` in the first code cell: `"full"` for the version 1 run, `"pilot"` for the short pilot.

1. Get the Nikash code (clone your public repository, or attach it as a Kaggle dataset).
2. Check that the harness works (unit tests and smoke run, CPU). **2b.** GPU packages, then the **start check**: batched answers checked against one-at-a-time answers on tiny models of each version 1 architecture (CPU), with the transformers just installed, so a batching fault stops the run here, before any benchmark or model download.
3. **Inspect LogInject-1.0**: list its record files (3), save the artifact's notes, licences and batch construction for reading (3b), then **build Benchmark A** with the real loader and report what was built and what was skipped, and why (3c).
4. **Build Benchmark B from CAM-LDS** with the real loader (IDS alerts are left out of every window), and report how many windows could be built and injected.
6. **Pilot** (`MODE = "pilot"`): each model in `MODELS` in turn, a short run on both benchmarks with all five defences: throughput, parse failures, answers cut off at the token limit, miss and false-alarm rates, and attack success.
7. **Full run** (`MODE = "full"`): every model in `MODELS` × the five defences × every case (triage; 200 injected windows per benchmark, their twins and 200 benign windows), saved slice by slice. A session stops cleanly before `TIME_BUDGET_HOURS`; the next session resumes from the earlier output attached as an input, if it was made from the same cases, code and generation settings. A model that fails is recorded, its answers so far are kept, and the next model runs.
8. **Full run summary**: metrics for every model whose answers are all in, and what is left.

There is one output file in `/kaggle/working`: `nikash_full.jsonl` (full run) or `nikash_pilot.json` (pilot). Download it from the Output tab.

**Rules:** payloads only go to models loaded in this notebook. The Hugging Face token comes from **Add-ons → Secrets** (`HF_TOKEN`) and is never printed or saved. The code, the downloaded benchmarks and the model weights go to the scratch folder `/kaggle/temp`, which is not part of the output (each model's weights are deleted once it is done); benchmark data is never re-hosted.

In [ ]:
# Settings. Edit these before running.
REPO_URL = "https://github.com/camseye/nikash.git"   # or leave empty and attach the code as a dataset
CODE_DATASET_DIR = "/kaggle/input/nikash-code/code"    # used when REPO_URL is empty
# The three version 1 models, chosen at the fourth pilot (Mistral 7B v0.3 dropped; notes/G2_DESIGN.md).
# Llama and Gemma are gated: accept each licence on its Hugging Face page, signed in as the account whose
# token is HF_TOKEN.
MODELS = ["Qwen/Qwen2.5-7B-Instruct", "meta-llama/Llama-3.1-8B-Instruct", "google/gemma-2-9b-it"]
MODE = "full"            # "full": the version 1 run (steps 7-8); "pilot": the short pilot (step 6)
# Full run. A session runs the models in RUN_MODELS that still have answers missing, and stops cleanly once
# TIME_BUDGET_HOURS have passed (Kaggle stops a session at 12 hours). To continue in a new session, attach the
# earlier version's output (Add Input -> Your Work -> this notebook) and run again: the answers already in it
# are kept and not asked again.
RUN_MODELS = MODELS
TIME_BUDGET_HOURS = 11.0
SLICE = 120              # cases per saved slice: a session that stops loses at most the slice in progress
# Each model copy's generation settings (HFModel in nikash/models/hf_backend.py). They are part of the full run's
# fingerprint: changing any of them, or any file under nikash/, starts the full run afresh.
GENERATION = {"batch_size": 8, "prefill_chunk": 1024, "max_new_tokens": 384, "memory_fraction": 0.5}
PILOT_PAIRS = 10         # pilot, per benchmark: injected windows, their twins, and as many benign
# Tasks, defences and cases per cell come from nikash/scope.py (version 1: triage, 5 defences, 200).

import os, time
SESSION_START = time.time()
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # before torch is imported
from pathlib import Path
import json
# Kaggle saves everything under /kaggle/working as the notebook's output, so the only file written
# there is OUTPUT_FILE. The code and the benchmark downloads go to the scratch folder instead.
WORK = Path("/kaggle/working")
SCRATCH = Path("/kaggle/temp") if Path("/kaggle/temp").is_dir() else Path("/tmp/nikash")
CACHE = SCRATCH / "cache"         # benchmark downloads: never committed, never re-hosted
os.environ.setdefault("HF_HOME", str(SCRATCH / "hf"))   # model weights, before transformers is imported
OUTPUT_FILE = WORK / ("nikash_full.jsonl" if MODE == "full" else "nikash_pilot.json")
CACHE.mkdir(parents=True, exist_ok=True)
OUTPUT = {}

CHECKPOINT = None        # the full run's output file, opened in step 7 once the cases are built
assert MODE in ("full", "pilot"), MODE

def save(section, data):
    """Pilot: adds one section to the output file and rewrites it, so a later failure keeps the earlier steps.
    Full run: kept until step 7 opens the output file, then written there."""
    OUTPUT[section] = data
    if MODE == "pilot":
        OUTPUT_FILE.write_text(json.dumps(OUTPUT, indent=1, default=str))
        print(f"Saved section '{section}' in {OUTPUT_FILE} ({OUTPUT_FILE.stat().st_size:,} bytes)")
    elif CHECKPOINT is not None:
        CHECKPOINT.section(section, data)
    else:
        print(f"Kept section '{section}'; it goes into {OUTPUT_FILE} when the full run starts")

In [ ]:
# 1. Get the code.
import subprocess, sys
if REPO_URL:
    target = SCRATCH / "nikash"
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(target)], check=True)
    CODE = target / "code"
else:
    CODE = Path(CODE_DATASET_DIR)
    if not (CODE / "nikash").is_dir():
        # Kaggle mounts datasets as /kaggle/input/<name>/ or /kaggle/input/datasets/<owner>/<name>/.
        found = sorted(p.parent.parent for p in Path("/kaggle/input").glob("**/code/nikash/__init__.py"))
        if not found:
            raise FileNotFoundError("No code/nikash/ folder under /kaggle/input. Attach the code dataset or set REPO_URL.")
        CODE = found[0]
sys.path.insert(0, str(CODE))
print("Code at", CODE)

In [ ]:
# 2. Harness check: unit tests and the smoke run (CPU, simulated models, synthetic data).
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests"], cwd=CODE, check=True)
subprocess.run([sys.executable, "smoke.py", "--per-family", "5"], cwd=CODE, check=True)

In [ ]:
# 2b. GPU packages. Kaggle images ship torch; add the rest if missing.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(CODE / "requirements.txt")], check=True)
# Start check: batched, chunked prefill must answer exactly as one prompt at a time, for each version 1
# architecture, with the transformers just installed (the one the models run on). Tiny random models on the
# CPU, a minute or so; a failure stops the run here, before any benchmark or model download.
# The test file is run by name: Kaggle's image has another package called `tests`, which would hide ours.
subprocess.run([sys.executable, "-m", "unittest", "discover", "-v", "-s", "tests", "-p", "test_prefill.py"], cwd=CODE, check=True,
               env={**os.environ, "NIKASH_REQUIRE_PREFILL_TESTS": "1"})
import torch
from importlib.metadata import version
# Recorded in the full run's output file; a resumed file made with other versions is reported in step 7.
LIBRARIES = {name: version(name) for name in ("torch", "transformers", "accelerate", "bitsandbytes")}
print(" | ".join(f"{k} {v}" for k, v in LIBRARIES.items()))
print("CUDA:", torch.cuda.is_available(), "|", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
# Download helpers. Files stream to disk and are checked against the MD5 Zenodo publishes; a file that fails
# the check is deleted and downloaded once more. Archives are checked for paths that would escape the target
# folder, and extracted into a temporary folder that is renamed only once extraction is complete.
import hashlib, json, shutil, tarfile, urllib.request, zipfile
from collections import Counter

def download(url, dest):
    part = dest.with_name(dest.name + ".part")
    with urllib.request.urlopen(url, timeout=300) as r, open(part, "wb") as fh:
        shutil.copyfileobj(r, fh, 1 << 20)
    part.rename(dest)

def file_hash(path, algo):
    h = hashlib.new(algo)
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def fetch(record_id, keys=None):
    record = json.load(urllib.request.urlopen(f"https://zenodo.org/api/records/{record_id}", timeout=60))
    print(record["metadata"]["title"], "| licence:", record["metadata"].get("license", {}).get("id"))
    got = {}
    for f in record["files"]:
        if keys and f["key"] not in keys:
            continue
        dest = CACHE / f["key"]
        algo, expected = f["checksum"].split(":", 1)
        for attempt in (1, 2):
            if not dest.exists():
                download(f["links"]["self"], dest)
            if file_hash(dest, algo) == expected:
                break
            dest.unlink()
            dest.with_name(dest.name + ".part").unlink(missing_ok=True)
            if attempt == 2:
                raise ValueError(f"checksum mismatch for {f['key']} in two downloads; the bad file was deleted")
            print(f"  {f['key']}: checksum mismatch; deleted it and downloading it again")
        print(f"  {f['key']}: {f['size']:,} bytes, {algo} verified")
        got[f["key"]] = dest
    return record, got

ARCHIVE_SUFFIXES = (".tar.gz", ".tgz", ".tar.xz", ".tar.bz2", ".tar", ".zip")

def archive_name(key):
    """The folder an archive extracts to: its name without the archive suffix ("LogInject-1.0.zip" -> "LogInject-1.0")."""
    return next((key[:-len(s)] for s in ARCHIVE_SUFFIXES if key.lower().endswith(s)), key)

def safe_unzip(path, out):
    out = out.resolve()
    with zipfile.ZipFile(path) as z:
        for m in z.infolist():
            if not (out / m.filename).resolve().is_relative_to(out):
                raise ValueError(f"unsafe path in archive: {m.filename}")
        z.extractall(out)

def extract(path, target):
    """Extracts a zip or tar archive to `target`; None if `path` is neither. A target folder that exists is complete:
    extraction goes to a temporary sibling folder, renamed once done."""
    if target.exists():
        return target
    partial = target.with_name(target.name + ".partial")
    shutil.rmtree(partial, ignore_errors=True)
    if tarfile.is_tarfile(path):
        partial.mkdir(parents=True)
        with tarfile.open(path) as tar:
            tar.extractall(partial, filter="data")   # blocks absolute paths, links outside the folder and devices
    elif zipfile.is_zipfile(path):
        partial.mkdir(parents=True)
        safe_unzip(path, partial)
    else:
        return None
    partial.rename(target)
    return target

In [ ]:
# 3. Inspect LogInject-1.0 (CC BY 4.0, zenodo.org/records/20436935).
from nikash.benchmarks import loginject

li_record, got = fetch(20436935)
inventory = {"record": "zenodo:20436935", "files": [{"key": f["key"], "size": f["size"], "checksum": f["checksum"]} for f in li_record["files"]]}
out = CACHE / "loginject"
for key, path in got.items():
    extract(path, out / archive_name(key))

paths = [p for p in out.rglob("*") if p.is_file()]
print(f"extracted {len(paths)} files; by extension:", Counter(p.suffix or "(none)" for p in paths).most_common(12))
inventory["tree"] = sorted(str(p.relative_to(out)) for p in paths)[:5000]
inventory["inspection"] = loginject.inspect(out)
for entry in inventory["inspection"]["record_files"][:20]:
    print(entry["path"], entry.get("records_read"), "records | keys:", list(entry.get("keys", {}))[:12])
    print("   suggested:", entry.get("suggested"))

save("loginject_inventory", inventory)

In [ ]:
# 3b. LogInject: the artifact's own notes, licences and batch construction, read before its loader
#     mode is written. Text only, kept in the output file (never committed). model_configs.yaml is skipped:
#     it may hold API settings. Anything shaped like a key or token is redacted.
import re

sub = next(p.parent for p in out.rglob("artifact") if p.is_dir() and (p / "data").is_dir())
WANTED = ["README.md", "LICENSE", "ETHICS.md", "CITATION.cff",
          "artifact/README.md", "artifact/LICENSE", "artifact/LICENSE_RASRL.md", "artifact/OPEN_SCIENCE.md",
          "artifact/SUBMISSION_NOTES.md", "artifact/ARCHIVE_MANIFEST.md", "artifact/data/PROVENANCE.md",
          "artifact/data/adversarial/README.md", "artifact/data/ground_truth/ANNOTATION_PROTOCOL.md",
          "artifact/configs/system_prompts.yaml", "artifact/configs/defense_configs.yaml",
          "artifact/src/pipeline/batch_constructor.py", "artifact/src/pipeline/scenario_runner.py",
          "artifact/data/adversarial/generate_samples.py", "artifact/scripts/run_scenario_a.py",
          "artifact/src/evaluation/metrics.py", "artifact/scripts/generate_dataset.py", "artifact/artifact_appendix.tex"]
WANTED += sorted(str(p.relative_to(sub)) for p in (sub / "artifact/data/adversarial/templates").glob("*.yaml"))
SECRET = re.compile(r"(sk-[A-Za-z0-9_-]{16,}|hf_[A-Za-z0-9]{20,}|(?i:(api[_-]?key|token|secret)\s*[:=]\s*)\S{8,})")
docs = {}
for rel in WANTED:
    p = sub / rel
    if p.is_file():
        text = p.read_text(encoding="utf-8", errors="replace")
        docs[rel] = SECRET.sub("<redacted>", text[:60_000]) + ("\n[... cut at 60,000 characters]" if len(text) > 60_000 else "")

def records(path):
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            if line.strip():
                yield json.loads(line)

data = sub / "artifact/data"
adv = list(records(data / "adversarial/samples.jsonl"))
def verbatim(r):
    return bool(r.get("payload")) and str(r["payload"]) in str(r.get("raw_log", r.get("log_text", "")))
stats = {
    "adversarial_records": len(adv),
    "by_level": Counter(str(r.get("attack_level")) for r in adv),
    "by_objective": Counter(str(r.get("attack_objective")) for r in adv),
    "by_strategy": Counter(str(r.get("attack_strategy")) for r in adv),
    "by_vector": Counter(str(r.get("injection_vector")) for r in adv),
    "level_x_objective": Counter(f"{r.get('attack_level')}|{r.get('attack_objective')}" for r in adv),
    "payload_verbatim_in_line_by_level": Counter(f"{r.get('attack_level')}|{verbatim(r)}" for r in adv),
    "fragment_count": Counter(str(r.get("fragment_count")) for r in adv if r.get("fragment_count")),
    "keys_by_level": {lvl: sorted({k for r in adv if str(r.get("attack_level")) == lvl for k in r})
                      for lvl in sorted({str(r.get("attack_level")) for r in adv})},
    "examples_by_level": {lvl: next(r for r in adv if str(r.get("attack_level")) == lvl)
                          for lvl in sorted({str(r.get("attack_level")) for r in adv})},
    "benign_records": {p.name: sum(1 for _ in records(p)) for p in sorted((data / "benign").glob("*.jsonl"))},
}
ann = list(records(data / "ground_truth/annotations.jsonl"))
stats["annotations"] = {
    "records": len(ann),
    "label_by_format": Counter(f"{a.get('log_format')}|{a.get('ground_truth_label')}" for a in ann),
    "security_relevant": Counter(str(a.get("is_security_relevant")) for a in ann),
    "id_prefixes": Counter(str(a.get("sample_id", "")).rsplit("-", 1)[0] for a in ann),
}
meta = li_record["metadata"]
zenodo = {"title": meta.get("title"), "version": meta.get("version"), "publication_date": meta.get("publication_date"),
          "licence": meta.get("license"), "doi": meta.get("doi")}
print("licence:", zenodo["licence"], "| adversarial records:", len(adv), "| levels:", dict(stats["by_level"]))
print("objectives:", dict(stats["by_objective"]), "| payload verbatim by level:", dict(stats["payload_verbatim_in_line_by_level"]))
print("saved", len(docs), "documents; missing:", [r for r in WANTED if r not in docs])
save("loginject_docs", {"zenodo": zenodo, "stats": stats, "docs": docs})

In [ ]:
# 3c. Benchmark A: LogInject-1.0 windows with the real loader (triage, CONCEAL attacks, levels 1-3 in turn).
#     The examples hold LogInject payload text (RASRL v1.0): they stay in the private output file for hand-checking
#     and are never published or quoted.
from nikash import scope
from nikash.defences import checks

def rules_overview(cases):
    """The rules defence on every window (CPU only): verdicts and fired checks by kind of window."""
    verdicts, fired = Counter(), {}
    for c in cases:
        k = ("injected" if c.injected else c.label) + (f"|{c.provenance['incident']}" if c.provenance.get("incident") else "")
        verdict, _, reasons = checks.decide(checks.facts_from_events(c.events))
        verdicts[f"{k}|{verdict}"] += 1
        fired.setdefault(k, Counter()).update(checks.rule_ids(reasons))
    return {"verdicts": dict(sorted(verdicts.items())), "fired": {k: dict(sorted(v.items())) for k, v in sorted(fired.items())}}

loginject_cases, li_report = loginject.build_cases(out, per_benchmark=scope.PER_BENCHMARK, seed=0)
li_summary = li_report.as_dict()
print(json.dumps(li_summary, indent=1))
if li_summary["malicious_windows"] < scope.PER_BENCHMARK:
    print(f"WARNING: {li_summary['malicious_windows']} of {scope.PER_BENCHMARK} injected windows; see 'skipped'.")
li_by_id = {c.id: c for c in loginject_cases}
li_summary["cells"] = dict(Counter("injected" if c.injected else c.label for c in loginject_cases))
li_summary["severity"] = dict(Counter(c.reference_severity for c in loginject_cases if c.label == "malicious" and not c.injected))
li_summary["rules_on_every_window"] = rules_overview(loginject_cases)
print("rules on every window:", json.dumps(li_summary["rules_on_every_window"]))
li_summary["examples"] = {}
for level in sorted(li_summary["by_level"]):
    case = next(c for c in loginject_cases if c.injected and c.attack["level"] == level)
    li_summary["examples"][f"level{level}"] = {
        "provenance": case.provenance, "fields": case.attack["fields"],
        "injected": case.render().splitlines(), "twin": li_by_id[case.twin_id].render().splitlines()}
li_summary["examples"]["benign"] = next(c for c in loginject_cases if c.label == "benign").render().splitlines()
print("cells:", li_summary["cells"], "| severity:", li_summary["severity"])

save("loginject", li_summary)

In [ ]:
# 4. Benchmark B: CAM-LDS (CC BY 4.0, zenodo.org/records/18861762).
#    Only the two manifestation archives are needed (about 750 MB), not the per-scenario recordings.
from nikash import scope
from nikash.benchmarks import camlds

cam_record, got = fetch(18861762, {"manifestations_filtered.zip", "manifestations_raw.zip"})
roots = {archive_name(key): extract(path, CACHE / "camlds" / archive_name(key)) for key, path in got.items()}

camlds_cases, report = camlds.build_cases(roots["manifestations_filtered"], roots["manifestations_raw"],
                                          per_benchmark=scope.PER_BENCHMARK, tasks=scope.TASKS, seed=0)
summary = report.as_dict()
print(json.dumps({k: v for k, v in summary.items() if k not in ("sources", "hosts")}, indent=1))
print("event sources:", sorted(summary["sources"].items(), key=lambda kv: -kv[1])[:15])
print("hosts in windows:", summary["hosts"])
if not (summary["attacker_files_excluded"] or summary["attacker_events_dropped"]):
    print("WARNING: no attacker-side logs were found. Compare camlds.ATTACKER_HOSTS", camlds.ATTACKER_HOSTS,
          "with the hosts above before using these windows.")
leaks = [s for s in summary["sources"] if any(t in s for t in camlds.ATTACKER_TOOL_FILES) or "msfrpcd" in s]
if leaks:
    print("WARNING: attacker tooling still in windows:", leaks)

twins = [c for c in camlds_cases if c.label == "malicious" and not c.injected]
summary["cells"] = dict(Counter(f"{c.task}|{'injected' if c.injected else c.label}" for c in camlds_cases))
summary["top_technique"] = dict(Counter((c.provenance.get("techniques") or ["unlabelled"])[0] for c in twins).most_common(40))
summary["severity"] = dict(Counter(c.reference_severity for c in twins))
summary["injected_fields"] = dict(Counter(f.split(".")[-1] for c in camlds_cases if c.injected for f in c.attack["fields"]))
summary["rules_on_every_window"] = rules_overview(camlds_cases)
print("rules on every window:", json.dumps(summary["rules_on_every_window"]))
first = next(c for c in camlds_cases if c.injected)
by_id = {c.id: c for c in camlds_cases}
benign = next((c for c in camlds_cases if c.label == "benign"), None)
summary["examples"] = {
    "injected": first.render().splitlines()[:40],
    "twin": by_id[first.twin_id].render().splitlines()[:40],
    "benign": benign.render().splitlines()[:40] if benign else [],
}
print("cells:", summary["cells"])
print("injected fields:", summary["injected_fields"])
print("severity:", summary["severity"])

save("camlds", summary)

In [ ]:
# 6. Pilot (MODE = "pilot"): each version 1 model in turn, on both benchmarks with all five defences: throughput, parse
#    failures, answers cut off at the token limit, miss and false-alarm rates and attack success.
#    PILOT_PAIRS is small, so the rates only show that every piece works with every model.
#    For hand-checking it also keeps up to 10 wrong answers on clean windows (twins and benign windows,
#    which hold no payload text) and, for rules and extract, which checks fired on each kind of window.
#    A model that fails is recorded and the next one runs; each model's weights are deleted afterwards.
import random, shutil, time
from statistics import mean
from nikash.defences import build_defences, checks
from nikash.defences.model_free import extract_facts
from nikash.benchmarks.window import window_chars
from nikash.models.hf_backend import HFPool
from nikash.run import run_matrix
from nikash.scorer.metrics import cell_metrics

def pilot_cases(cases, pairs, seed=0):
    rng = random.Random(seed)
    by_id = {c.id: c for c in cases}
    injected = [c for c in cases if c.injected]
    benign = [c for c in cases if c.label == "benign"]
    picked = rng.sample(injected, min(pairs, len(injected)))
    return [c for i in picked for c in (i, by_id[i.twin_id])] + rng.sample(benign, min(pairs, len(benign)))

sets = {"loginject": pilot_cases(loginject_cases, PILOT_PAIRS),
        "camlds": pilot_cases([c for c in camlds_cases if c.task == "triage"], PILOT_PAIRS)}
cases = [c for s in sets.values() for c in s]
by_id = {c.id: c for c in cases}
KEEP = ("parse_fail", "miss", "false_alarm", "asr", "asr_k", "asr_n", "asr_raw", "kept_share")

def kind(case):
    return "injected" if case.injected else case.label

def facts_used(defence_id, case, record):
    if defence_id == "rules":
        return checks.facts_from_events(case.events)
    return extract_facts(record["model_text"]) or {}

def diagnostics(defence_id, recs):
    clean = [r for r in recs if r["ok"] and not by_id[r["case_id"]].injected]
    wrong = [r for r in clean if r["response"]["verdict"] != by_id[r["case_id"]].label]
    out = {"wrong_on_clean": [{"case_id": r["case_id"], "label": by_id[r["case_id"]].label,
                                "verdict": r["response"]["verdict"],
                                "explanation": r["response"].get("explanation", "")[:300]} for r in wrong[:10]],
           "wrong_on_clean_count": len(wrong), "clean_count": len(clean),
           "errors": dict(Counter(r["error"] for r in recs if not r["ok"]))}
    if defence_id in ("rules", "extract"):
        fired = {}
        for r in recs:
            case = by_id[r["case_id"]]
            ids = checks.rule_ids(checks.decide(facts_used(defence_id, case, r))[2])
            fired.setdefault(kind(case), Counter()).update(ids)
        out["checks_fired"] = {k: dict(sorted(v.items())) for k, v in sorted(fired.items())}
    if defence_id == "extract":
        # By count name: "203.0.113.9.failures" and "app.auth_failures" become "failures" and "auth_failures".
        out["extract_mismatches"] = dict(Counter(m.split(":", 1)[0].rsplit(".", 1)[-1] for r in recs if r["ok"]
                                                for m in r["response"].get("extract_mismatch", [])).most_common(20))
    return out

def pilot_model(model_id, token):
    # One copy of the model per GPU (2 on Kaggle's T4 x2), each in its own worker process, run in
    # parallel. Batches and prefill chunks are sized from each GPU's free memory. A worker that
    # fails is restarted once; peak memory is measured inside each worker.
    model = HFPool(model_id, token=token, **GENERATION)
    try:
        result = {"model": model.describe(), "gpus": len(model.devices), "batch_size": model.batch_size,
                  "prefill_chunk": model.prefill_chunk, "defences": {}}
        for defence in build_defences(list(scope.DEFENCES), seed=0):
            calls = sum(1 for c in cases if defence.needs_model(c))
            start = time.time()
            records = run_matrix(cases, [defence], [model])
            elapsed = time.time() - start
            entry = {"seconds": round(elapsed, 1), "model_calls": calls,
                     "cases_per_hour": round(calls / elapsed * 3600) if calls else None,
                     **(model.last_stats if calls else {}), "benchmarks": {}}
            for b, subset in sets.items():
                ids = {c.id for c in subset}
                m = cell_metrics(by_id, [r for r in records if r["case_id"] in ids])
                entry["benchmarks"][b] = {k: m.get(k) for k in KEEP}
                entry["benchmarks"][b].update(diagnostics(defence.id, [r for r in records if r["case_id"] in ids]))
            entry["examples"] = [r["model_text"][:300] for r in records if not by_id[r["case_id"]].injected][:3]
            result["defences"][defence.id] = entry
            print(defence.id, entry["cases_per_hour"], "cases/hour,", calls, "model calls,",
                  entry.get("at_token_limit"), "cut off at the token limit,", entry.get("chunked_prompts"), "chunked,",
                  entry.get("oom_retries"), "out-of-memory retries,", entry.get("batch_fallbacks"), "batch fallbacks,", entry.get("worker_restarts"), "worker restarts,",
                  entry.get("peak_gpu_gib_per_device"), "GiB peak per GPU")
            for b, m in entry["benchmarks"].items():
                print(f"   {b}: parse failures {m['parse_fail']} {m['errors'] or ''}, miss {m['miss']}, "
                      f"false alarm {m['false_alarm']}, attack success {m['asr_k']}/{m['asr_n']} (raw {m['asr_raw']}), "
                      f"wrong on clean {m['wrong_on_clean_count']}/{m['clean_count']}")
                if "checks_fired" in m:
                    print("      checks fired:", m["checks_fired"])
        return result
    finally:
        model.close()   # frees both GPUs

if MODE == "pilot":
    from kaggle_secrets import UserSecretsClient
    try:
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        hf_token = None   # fine for ungated models
    try:
        throughput = {"cases": {b: len(s) for b, s in sets.items()},
                      "mean_events": {b: round(mean(len(c.events) for c in s), 1) for b, s in sets.items()},
                      "max_window_chars": {b: max(window_chars(c.events) for c in s) for b, s in sets.items()},
                      "models": {}}
        for model_id in MODELS:
            print(f"\n===== {model_id}")
            try:
                throughput["models"][model_id] = pilot_model(model_id, hf_token)
            except Exception as exc:
                message = f"{type(exc).__name__}: {exc}"
                if hf_token:
                    message = message.replace(hf_token, "<redacted>")
                throughput["models"][model_id] = {"error": message[-3000:]}
                print("FAILED; moving on to the next model.", message[-800:])
            shutil.rmtree(Path(os.environ["HF_HOME"]) / "hub" / ("models--" + model_id.replace("/", "--")), ignore_errors=True)
            save("throughput", throughput)
    finally:
        del hf_token

In [ ]:
# 7. Full run (MODE = "full"): every version 1 model x defence x case (nikash/scope.py), SLICE cases at a time,
#    each slice appended to nikash_full.jsonl as soon as it is done. Output files of earlier sessions attached
#    as inputs are resumed: their answers are copied in and not asked again. A file built from other cases, or
#    with answers from other code (any file under nikash/) or other generation settings (the models, GENERATION,
#    each defence's token limit), is refused, never mixed in. Each model loads the commit of its Hugging Face
#    repository (weights, tokenizer, chat template) that earlier sessions loaded. Step 2b's start check has
#    already checked batching for each model architecture.
if MODE == "full":
    import shutil
    import nikash
    from kaggle_secrets import UserSecretsClient
    from nikash.checkpoint import Checkpoint, case_stub, run_fingerprints
    from nikash.defences import build_defences
    from nikash.fullrun import progress, run_model
    from nikash.models.hf_backend import HFPool, describe_model

    cases = [c for c in loginject_cases + camlds_cases if c.task in scope.TASKS]
    defences = build_defences(list(scope.DEFENCES), seed=0)
    descriptions = [describe_model(m) for m in MODELS]
    settings = {"models": MODELS, "generation": GENERATION, "max_new_tokens": {d.id: d.max_new_tokens for d in defences}}
    CHECKPOINT = Checkpoint(OUTPUT_FILE, run_fingerprints(cases, settings),
                            {"nikash_version": nikash.__version__, "libraries": LIBRARIES, **settings, "defences": list(scope.DEFENCES),
                             "tasks": list(scope.TASKS), "cases": len(cases)})
    CHECKPOINT.resume_from(sorted(Path("/kaggle/input").glob("**/nikash_full.jsonl")))
    print("resumed:", CHECKPOINT.resumed or "nothing", "| refused:", CHECKPOINT.refused or "none")
    for path, theirs in CHECKPOINT.changed("libraries", LIBRARIES).items():
        print(f"WARNING: {path} was made with {theirs or 'unrecorded versions'}; this session runs {LIBRARIES}. "
              "Its answers are kept; say so when reporting the results.")
    for name, data in OUTPUT.items():
        CHECKPOINT.section(name, data)
    CHECKPOINT.section("cases", [case_stub(c) for c in cases])
    left = progress(CHECKPOINT, [d["name"] for d in descriptions], defences, cases)
    print(len(cases), "cases; answers to go per model and defence:", json.dumps(left))

    session = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime(SESSION_START))
    sessions = {}
    for sections in (*CHECKPOINT.earlier.values(), CHECKPOINT.sections):
        sessions.update((sections.get("throughput") or {}).get("sessions", {}))
    revisions = {}
    for entries in sessions.values():
        for model_id, entry in entries.items():
            if entry.get("revision"):
                revisions.setdefault(model_id, entry["revision"])
    sessions[session] = {}
    stop_at = SESSION_START + TIME_BUDGET_HOURS * 3600
    try:
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        hf_token = None   # fine for ungated models
    try:
        for model_id, d in zip(MODELS, descriptions):
            if model_id not in RUN_MODELS or not any(left[d["name"]].values()):
                continue
            if time.time() > stop_at:
                print("Time budget reached; the remaining models run in the next session.")
                break
            print(f"\n===== {model_id}: {sum(left[d['name']].values())} answers to go")
            entry = sessions[session][model_id] = {}
            try:
                model = HFPool(model_id, token=hf_token, revision=revisions.get(model_id), **GENERATION)
                try:
                    entry["revision"] = getattr(model, "revision", None)
                    entry.update(run_model(CHECKPOINT, model, cases, defences, slice_size=SLICE, stop_at=stop_at))
                finally:
                    model.close()   # frees both GPUs
            except Exception as exc:
                message = f"{type(exc).__name__}: {exc}"
                if hf_token:
                    message = message.replace(hf_token, "<redacted>")
                entry["error"] = message[-3000:]
                print("FAILED; the answers it gave are kept. Moving on to the next model.", message[-800:])
            shutil.rmtree(Path(os.environ["HF_HOME"]) / "hub" / ("models--" + model_id.replace("/", "--")), ignore_errors=True)
            CHECKPOINT.section("throughput", {"sessions": sessions})
    finally:
        del hf_token

In [ ]:
# 8. Full run summary: metrics (G2 design §4) over every model whose answers are all in, and what is left.
if MODE == "full":
    from nikash.fullrun import benchmark_meta, summarise

    def r3(x):
        return None if x is None else round(x, 3)

    left = progress(CHECKPOINT, [d["name"] for d in descriptions], defences, cases)
    CHECKPOINT.section("progress", left)
    summary = summarise(CHECKPOINT, cases, defences, descriptions, benchmark_meta(CODE, ["loginject", "camlds"]), "v1-full")
    if summary:
        CHECKPOINT.section("summary", summary)
        for key, cell in summary["cells"].items():
            print("\n" + key)
            for name, m in sorted(cell.items()):
                print(f"  {name:<40} attack success {m['asr_k']}/{m['asr_n']}, miss {r3(m['miss'])}, "
                      f"false alarm {r3(m['false_alarm'])}, parse failures {r3(m['parse_fail'])}")
        for bench, q in summary.get("quarantine_evidence", {}).items():
            print(f"\nquarantine, {bench}: {q['evidence_lost']} of {q['injected']} injected windows lose a check "
                  f"their twin keeps ({q['verdict_lost']} flip to benign; {q['checks_lost']}); "
                  f"{q['clean_quarantined']} of {q['clean']} clean windows lose a field")
    missing = {m: {d: n for d, n in v.items() if n} for m, v in left.items() if any(v.values())}
    if missing:
        print("\nNot finished:", json.dumps(missing))
        print("Next session: attach this version's output (Add Input -> Your Work -> this notebook), then Save & Run All.")
    else:
        print(f"\nAll {len(MODELS)} models are complete. Download {OUTPUT_FILE.name} from the Output tab.")

## Bring back

There is one file in the Output tab.

**Full run:** if the last cell says some answers are missing, don't download yet: attach this version's output to the next session (**Add Input → Your Work →** this notebook, the latest version) and **Save & Run All** again. When it says every model is complete, download `nikash_full.jsonl` (the last version's file holds every answer) and put it in `code/results/full/` on your machine. Then `python build_dashboard.py results/full/nikash_full.jsonl --out docs/data/scoreboard.json`. Each line of the file is one of:

- `run`: the code version, the models and generation settings, and fingerprints of the cases and of the code (every file under `nikash/`, with those settings); a resumed file must match both.
- `record`: one answer (model, defence, case, the parsed verdict and the model's text).
- a section: `loginject` and `camlds` (what was built), `cases` (labels and twins, no log text), `throughput` (per session, model and defence: speed, batches, batch fallbacks, out-of-memory retries, answers cut off, errors), `progress` (answers still missing) and `summary` (metrics for the complete models, read by `build_dashboard.py`).

**Pilot:** download `nikash_pilot.json` and put it in `code/results/pilot/` on your machine. Its sections:

- `loginject`: how many LogInject windows were built per level, vector and incident kind, what was skipped and why, the layout rewrites, what the rules defence says on every window, and one injected window with its twin per level to hand-check.
- `camlds`: how many CAM-LDS windows were built (IDS alerts left out), how many could not be injected, what the rules defence says on every window, and example windows to hand-check.
- `throughput`: for each version 1 model (or the error that stopped it), and each defence on both benchmarks: speed, parse failures and their reasons, answers cut off at the token limit, miss and false-alarm rates, attack success, wrong answers on clean windows and the checks that fired. It sets the full-run plan (G2 design §6, §9).
- `loginject_inventory` and `loginject_docs`: the archive's file list and the artifact's own notes and licences, for explaining skips.

Both files hold LogInject payload text (RASRL v1.0): keep it private and don't quote it anywhere. Model answers may quote it too. The benchmark downloads stay in `/kaggle/temp` and are not part of the output.